## Aim 

Compute the betweeness centrality for the entire Bangalore.

## Reading in the Python libraries

In [ ]:
from pathlib import Path

import ee
import geemap
import geopandas as gpd
import momepy
import numpy as np
import osmnx as ox
import pandas as pd
from matplotlib import pyplot as plt
import rasterio
from rasterio.features import rasterize
from scipy.stats import linregress, pearsonr
from tqdm import tqdm

In [ ]:
cloud_project = 'ee-ellenarun2'

try:
    ee.Initialize(project=cloud_project)
except:
    ee.Authenticate()
    ee.Initialize(project=cloud_project)

## Loading the data

In [ ]:
BASE_DIR = Path.cwd().parents[1]
print(BASE_DIR)

DATA_DIR = BASE_DIR / "data"
IMAGES_DIR = BASE_DIR / "images"
PROCESSED_DIR = DATA_DIR / "processed"
## LOGS_DIR = BASE_DIR / "logs"

path_graphml = DATA_DIR / "graphml_bang" / "bangalore_graphml.graphml"

# WARDS_GPKG_DIR = DATA_DIR / "gpkgs"
# WARDS_G_DIR =  DATA_DIR / "g_files"

Loading the ward data:

In [ ]:
## Detailed wards
df_inpath = PROCESSED_DIR / "bang_fix_mapshaper.shp"
print(df_inpath)

In [ ]:
df = gpd.read_file(df_inpath)
df.head()

In [ ]:
G = ox.io.load_graphml(path_graphml)

## Selecting the Bellandur area

Looking at the map, the Bellandur area consists of three wards:

- 46 - Yamalur
- 47 - Bellanduru
- 49 - Shivanasamudra ward

Selecting the wards:

In [ ]:
bellandur = df[df["ward_name"].isin(["46 - Yamalur", "47 - Bellanduru", "49 - Shivanasamudra Ward"])]
bellandur

In [ ]:
bellandur.plot()

In [ ]:
## Save this
bellandur_wards_path = PROCESSED_DIR / "bellandur.shp"
bellandur.to_file(bellandur_wards_path)

In [ ]:
# Merge these wards together
bellandur_dissolved = bellandur.dissolve()
bellandur_dissolved.plot()

In [ ]:
## Save this
dissolved_bang_path = PROCESSED_DIR / "bellandur_dissolved.shp"
bellandur_dissolved.to_file(dissolved_bang_path)

In [ ]:
bellandur_geometry = bellandur_dissolved.geometry.iloc[0]
bellandur_geometry

In [ ]:
## Extract the network:
G_bellandur = ox.truncate.truncate_graph_polygon(G, bellandur_geometry)

In [ ]:
fig, ax = ox.plot_graph(G_bellandur, node_size=0, edge_color="w", edge_linewidth=0.2)

In [ ]:
streets_graph = ox.projection.project_graph(G_bellandur)


In [ ]:
edges = ox.graph_to_gdfs(
    ox.convert.to_undirected(streets_graph),
    nodes=False,
    edges=True,
    node_geometry=False,
    fill_edge_geometry=True,
    )

In [ ]:
## https://docs.momepy.org/en/latest/api/momepy.betweenness_centrality.html
## https://docs.momepy.org/en/v0.7.2/user_guide/graph/centrality.html

In [ ]:
streets_graph = ox.projection.project_graph(G_bellandur)

In [ ]:
edges = ox.graph_to_gdfs(
    ox.convert.to_undirected(streets_graph),
    nodes=False,
    edges=True,
    node_geometry=False,
    fill_edge_geometry=True,
    )

In [ ]:
f, ax = plt.subplots(figsize=(10, 10))
edges.plot(ax=ax, linewidth=0.4)
ax.set_axis_off()
plt.show()

In [ ]:
primal = momepy.gdf_to_nx(edges, approach="primal")

In [ ]:
primal = momepy.betweenness_centrality(primal, name='betweenness_metric_e', mode='edges', weight='mm_len')

In [ ]:
primal_gdf = momepy.nx_to_gdf(primal, points=False)

In [ ]:
f, ax = plt.subplots(figsize=(15, 15))
primal_gdf.plot(ax=ax, column='betweenness_metric_e', cmap='Spectral_r', scheme='quantiles', alpha=0.6, legend = True)
ax.set_axis_off()
ax.set_title('betweennes edge based')
plt.show()

In [ ]:
fig_path = IMAGES_DIR / 'betweeness_all_roads_bellandur.png'
f, ax = plt.subplots(figsize=(15, 15))

primal_gdf.plot(
    ax=ax,
    column='betweenness_metric_e',
    cmap='Spectral_r',
    scheme='quantiles',
    k=5,
    legend=True,
    alpha=0.6,
    legend_kwds={
        "title": "Edge betweenness",
        "fmt": "{:.1e}",
        "loc": "lower left"
    }
)

ax.set_axis_off()
ax.set_title("Edge betweenness of Bellandur area", fontsize=14)
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
primal_gdf.shape

In [ ]:
primal_gdf.head()

In [ ]:
# Save to gpkg file (can view in QGIS if needed);
primal_gdf.crs

In [ ]:
## Convert it back to epsg 4326
primal_gdf_reproj = primal_gdf.to_crs(4326)

In [ ]:
## Save as a gpkg to read into QGIS:
path_betweeness = PROCESSED_DIR / 'betweeness.gpkg'
primal_gdf_reproj.to_file(path_betweeness)

## Land Surface temperature

In this script, the band 10 of Landsat 8 is used to compute the Land Surface Temperature. 

In [ ]:
print(dissolved_bang_path)
print(bellandur_wards_path)

In [ ]:
# https://geemap.org/notebooks/10_shapefiles/ 
# Convert to something geemap can understand
bellandur_fc = geemap.shp_to_ee(dissolved_bang_path)

In [ ]:
bellandur_wards = geemap.shp_to_ee(bellandur_wards_path)

In [ ]:
Map = geemap.Map()
Map.centerObject(bellandur_fc, zoom=13)
Map # Uncomment if you want to see the map. 
## Otherwise you can wait till later Maps as all layers are in them

In [ ]:
## Add the ward data
image = ee.Image().paint(bellandur_wards, 0, 2)
Map.addLayer(image, {'palette': 'red'}, "Bellandur wards")
Map

In [ ]:
# Map.addLayer(bellandur_wards, {'color': 'red'}, 'Bellandur three wards')
# Map

In [ ]:
Map.addLayer(bellandur_fc, {}, 'Bellandur outline')
Map

In [ ]:
colThsummer = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2') \
.filterBounds(bellandur_fc) \
.filterDate('2024-01-01', '2025-11-01') \
.filter(ee.Filter.calendarRange(3, 5, 'month'))

In [ ]:
## colThsummer

In [ ]:
def maskL8sr(image):
    ## Code obtained from:
    ## https://courses.spatialthoughts.com/end-to-end-gee-supplement.html#derive-lst-from-landsat-images 
    qaMask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturationMask = image.select('QA_RADSAT').eq(0)

    # Scaling factors
    opticalBands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
    thermalBands = image.select('ST_B.*').multiply(0.00341802).add(149.0)

    # Replace the original bands with the scaled ones and apply the masks.
    return image.addBands(opticalBands, None, True) \
    .addBands(thermalBands, None, True) \
    .updateMask(qaMask) \
    .updateMask(saturationMask)

In [ ]:
colThsummer = colThsummer.map(maskL8sr)

In [ ]:
## colThsummer

In [ ]:
imageThsummer = colThsummer.median()

In [ ]:
## imageThsummer

In [ ]:
image_clipped = imageThsummer.clip(bellandur_fc)

visualisation = {
  'bands': ['SR_B4', 'SR_B3', 'SR_B2'], 
  'min': 0,
  'max': 0.5,
  'gamma': [0.95, 1.1, 1]
}

Map.addLayer(image_clipped, visualisation, 'True Color Composite Bangalore')

Map

In [ ]:
thermalsummer = imageThsummer.select('ST_B10') \
.clip(bellandur_fc) \

Map.addLayer(thermalsummer, {
"min": 300,
"max": 320,
"palette": ['blue', 'white', 'red']},'Landsat_BT')

Map

In [ ]:
## Convert to Celsius
thermal_summer_c = (
thermalsummer
.clip(bellandur_fc)
.subtract(273.15)
.rename('LST_C')
)

In [ ]:
Map.addLayer(thermal_summer_c, {
    "min": 35,
    "max": 42,
    "palette": ['blue', 'white', 'red'],
},
'LST_Landsat')

vis_params = {'min': 35, 'max': 42, 'palette': ['blue', 'white', 'red']}
Map.add_colorbar(vis_params, label='Land Surface Temperature (°C)',position= 'bottomright')
Map

Export this map (for opening in QGIS):

In [ ]:
edges

In [ ]:
edges.shape

In [ ]:
## https://geemap.org/notebooks/11_export_image/#download-an-eeimage
## To view in QGIS:
filepath = PROCESSED_DIR / 'bellandur_lst.tif'
geemap.ee_export_image(
    thermal_summer_c, filename=filepath, 
    scale=30,              
    region= bellandur_fc.geometry(),
    file_per_band=False
)

In [ ]:
# Read back in the image (you can also convert to a raster):
import rasterio
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
primal_gdf_reproj.head()

In [ ]:
# Need to fix code a bit here
from rasterio.mask import mask

In [ ]:
with rasterio.open(filepath) as src:
    lst_masked, transform = mask(
        src,
        bellandur.geometry,
        crop=True
    )
    lst_masked = lst_masked[0]

In [ ]:
lst_masked = lst_masked.astype("float32")
lst_masked[lst_masked == src.nodata] = np.nan

In [ ]:
lst_masked[lst_masked <= -9999] = np.nan

In [ ]:
cmap = plt.cm.RdYlBu_r.copy()
cmap.set_bad(color=(0, 0, 0, 0))  # transparent


In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

im = ax.imshow(
    lst_masked,
    cmap=cmap,
    vmin=np.nanpercentile(lst_masked, 2),
    vmax=np.nanpercentile(lst_masked, 98),
    origin="upper"
)

plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
ax.axis("off")

plt.show()


In [ ]:
from rasterio.transform import array_bounds

height, width = lst_masked.shape
bounds = array_bounds(height, width, transform)
extent = [bounds[0], bounds[2], bounds[1], bounds[3]]


In [ ]:
lst_betweenness_path = IMAGES_DIR / 'lst_betweeness_bellandur.png'

In [ ]:
f, ax = plt.subplots(figsize=(15, 15))

im = ax.imshow(
    lst_masked,
    cmap=cmap,
    vmin=np.nanpercentile(lst_masked, 2),
    vmax=np.nanpercentile(lst_masked, 98),
    extent=extent,
    origin="upper"
)


primal_gdf_reproj.plot(
    ax=ax,
    column='betweenness_metric_e',
    cmap='Spectral_r',
    scheme='quantiles',
    k=5,
    legend=True,
    alpha=0.6,
    linewidth=1.2,
    legend_kwds={
        "title": "Edge betweenness",
        "fmt": "{:.1e}",
        "loc": "lower left"
    }
)


cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Land Surface Temperature")

ax.axis("off")
ax.set_title("Land Surface Temperature and Road Betweenness", fontsize=22)
plt.savefig(lst_betweenness_path, dpi=300, bbox_inches="tight")
plt.show()


## Matching LST pixels with betweeness values

In [ ]:
import rasterio
import geopandas as gpd
import numpy as np
from rasterio.features import rasterize
from rasterio.enums import MergeAlg


In [ ]:
with rasterio.open(filepath) as src:
    meta = src.meta.copy()
    transform = src.transform
    out_shape = (src.height, src.width)
    crs = src.crs

In [ ]:
print(f'The shape of the tif is {out_shape}. and the crs is {crs}')

In [ ]:
roads = gpd.read_file(path_betweeness)
## roads.plot() # quick check

In [ ]:
roads.head()

In [ ]:
# Check if the crs is the same between the raster and the shape file:
print(f'It is {roads.crs == crs} that the shape file of the raster LST and the geopandas file is the same')


In [ ]:
roads.shape

In [ ]:
# Create iterator
shapes = zip(roads.geometry, roads["betweenness_metric_e"])

In [ ]:
list(shapes)[:5]

In [ ]:
shapes = list(zip(roads.geometry, roads["betweenness_metric_e"]))

In [ ]:
shapes = list(zip(roads.geometry, roads["betweenness_metric_e"]))
print(len(shapes))  # must be > 0


In [ ]:
print(out_shape)
print(type(out_shape), type(out_shape[0]), type(out_shape[1]))


In [ ]:
# Step 1: rasterize roads one by one
betweenness_raster = np.full(out_shape, np.nan, dtype="float32")

for geom, val in shapes:
    temp = rasterize(
        [(geom, val)],
        out_shape=out_shape,
        transform=transform,
        fill=np.nan,
        all_touched=True,
        dtype="float32"
    )
    betweenness_raster = np.fmax(betweenness_raster, temp)


In [ ]:
betweenness_raster

In [ ]:
betweenness_raster.shape

In [ ]:
raster_betweeness_img_path = IMAGES_DIR / 'betweenness_raster_bellandur.png'

In [ ]:
# Assume betweenness_raster is your NumPy array
# Mask NaNs so they appear transparent
masked = np.ma.masked_invalid(betweenness_raster)

# Plot
fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(
    masked,
    cmap="YlOrRd",   # or "Spectral_r", "hot"
    origin="upper"
)

# Add colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("Road Betweenness", fontsize=14)

ax.set_title("Road betweenness raster", fontsize=18, fontweight="bold")
ax.axis("off")
plt.savefig(raster_betweeness_img_path, dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
## Export this to a .tif file to check in QGIS
betweenness_raster_path = PROCESSED_DIR / 'betweeness_raster_bellandur.tif'

# Update metadata for single-band raster above
meta.update({
    "dtype": "float32",
    "count": 1,
    "nodata": np.nan
})

# Write raster
with rasterio.open(betweenness_raster_path, "w", **meta) as dst:
    dst.write(betweenness_raster, 1)


Extract the LST raster and this only for the road pixels and plot: 

In [ ]:
raster_lst_betweeness_img_path = IMAGES_DIR / 'LST_roads_raster_bellandur.png'

In [ ]:
with rasterio.open(filepath) as src:
    lst = src.read(1).astype("float32")

# Mask NoData in LST
lst_masked = np.ma.masked_invalid(lst)

bet_masked = np.ma.masked_invalid(betweenness_raster)

lst_masked_for_roads = np.ma.masked_where(np.isnan(betweenness_raster), lst_masked)

fig, ax = plt.subplots(figsize=(10, 10))

vmin = 37
vmax = 41

im = ax.imshow(
    lst_masked_for_roads,
    cmap="coolwarm",  # LST color map
    origin="upper",
    vmin = vmin,
    vmax = vmax
)

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label("LST (°C)", fontsize=14)

ax.set_title("Land Surface Temperature only on road pixels", fontsize=18, fontweight="bold")
ax.axis("off")
plt.savefig(raster_lst_betweeness_img_path, dpi=300, bbox_inches='tight')
plt.show()

Overlay the two rasters (LST raster with LST pixelised betweenness centrality):

CODE TO DO

For the time being, the visualisation has happened in QGIS.

## Regression and correlation analysis

In [ ]:
lst_flat = lst_masked_for_roads.flatten()
bet_flat = betweenness_raster.flatten()

In [ ]:
mask = (~np.isnan(lst_flat)) & (~np.isnan(bet_flat))

In [ ]:
lst_valid = lst_flat[mask]
bet_valid = bet_flat[mask]
df = pd.DataFrame({
    "LST": lst_valid,
    "betweenness": bet_valid
})

df[0:-10]

In [ ]:
df.describe()

In [ ]:
## Remove outliers
df_clean = df[(df["LST"] != 0) & (df["betweenness"] != 0)]

In [ ]:
df_clean.describe()

In [ ]:
pearsonr(df_clean['LST'], df_clean['betweenness'])

In [ ]:
fig_path = IMAGES_DIR / 'regression_lst_betweenness_bellandur.png'

In [ ]:
x = df_clean['betweenness']
y = df_clean['LST']

In [ ]:
# linear regression
slope, intercept, r_value, p_value, std_err = linregress(x, y)

# regression line
x_line = np.linspace(x.min(), x.max(), 100)
y_line = intercept + slope * x_line

# plot
plt.figure(figsize=(7, 6))
plt.scatter(x, y, alpha=0.6, label="Wards")
plt.plot(x_line, y_line, color="red", label="Linear fit")

plt.xlabel("betweenness")
plt.ylabel("Land Surface Temperature (LST)")
plt.title("LST versus betweenness")
plt.legend()


# annotation


plt.text(
    0.05, 0.98,
    f"$R^2$ = {r_value**2:.2f}\np = {p_value:.2e}",
    transform=plt.gca().transAxes,
    verticalalignment="top"
)

plt.text(
    0.05, 0.90,
    f"$LST  = {intercept:.2f} {slope:.2f}\\,betweenness$",
    transform=plt.gca().transAxes,
    verticalalignment="top"
)
plt.savefig(fig_path, dpi=300, bbox_inches='tight')
plt.show()